<a href="https://colab.research.google.com/github/SourStone/Portafolio-Projectos-Programacion/blob/main/prototipo2_semillero.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import os
import re
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
import keras
import tensorflow
from tensorflow.keras.utils import to_categorical
from keras.models import Sequential, Input, Model
from keras.layers import Dense, Dropout, Flatten
from keras.layers import Conv2D, MaxPooling2D
from keras.layers.normalization import batch_normalization
from keras.layers.advanced_activations import LeakyReLU

In [ ]:
import PIL
import PIL.Image
import tensorflow_datasets as tfds
from tensorflow.keras import layers
import pathlib

acces drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls "/content/drive/My Drive/Colab Notebooks"

 defect_mnist.h5	       prototipo2_semillero.ipynb
 LeatherDefectClassification   sportimages.zip
 Practica_1.ipynb	      'sportimages.zip (Unzipped Files)'
 Practica_1_succesfull.ipynb   Untitled0.ipynb
 prototipo1_semillero.ipynb


lectura imagenes

metodo de lectura 2

parameters for the loader

In [ ]:
batch_size = 8
img_height = 227
img_width = 227

the loader

In [ ]:
#training data
data_dir = os.path.join(os.getcwd(), 'drive/My Drive/Colab Notebooks/LeatherDefectClassification')
train_ds = tensorflow.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split = 0.2,
    subset = "training",
    seed = 123,
    image_size=(img_height, img_width),
    batch_size = batch_size
)
val_ds = tensorflow.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

Found 3600 files belonging to 6 classes.
Using 2880 files for training.
Found 3600 files belonging to 6 classes.
Using 720 files for validation.


In [ ]:
#clases
class_names = train_ds.class_names
print(class_names)

['Folding marks', 'Grain off', 'Growth marks', 'loose grains', 'non defective', 'pinhole']


In [ ]:
#resize and rescale
IMG_SIZE = 180

resize_and_rescale = tensorflow.keras.Sequential([
    layers.Resizing(IMG_SIZE, IMG_SIZE),
    #layers.Rescaling(1./255)
])

In [ ]:
#augmentation
data_augmentation = tensorflow.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    #layers.RandomRotation(0.2),
])

data augmentation

In [ ]:

#augmentationt
AUTOTUNE = tensorflow.data.AUTOTUNE

def prepare(ds, shuffle=False, augment=False):
  #ds = ds.map(lambda x, y: (resize_and_rescale(x), y), num_parallel_calls=AUTOTUNE)

  if shuffle:
    ds = ds.shuffle(1000)

  #dont batch what has already been batched
  #ds = ds.batch(batch_size)

  if augment:
    ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)

  return ds.prefetch(buffer_size=AUTOTUNE)




aplly shuffle again and augmentation only on training set

In [ ]:
#train_ds = prepare(train_ds, shuffle=True, augment=True)
#val_ds = prepare(val_ds)

creating neural network

In [ ]:
from keras.backend import dropout
nClasses = 6
defect_model = tensorflow.keras.Sequential([
    tensorflow.keras.layers.Rescaling(1./255),
    tensorflow.keras.layers.Conv2D(32, 3, activation='relu'),
    tensorflow.keras.layers.MaxPooling2D((2,2)),
    tensorflow.keras.layers.Conv2D(32,3,activation='relu'),
    tensorflow.keras.layers.MaxPooling2D((2,2)),
    tensorflow.keras.layers.Conv2D(32, 3, activation='relu'),
    tensorflow.keras.layers.MaxPooling2D(),
    tensorflow.keras.layers.Flatten(),
    tensorflow.keras.layers.Dense(128, activation='relu'),
    tensorflow.keras.layers.Dense(128, activation='relu'),
    tensorflow.keras.layers.Dense(nClasses)
])

defect_model.compile(
    optimizer='adam',
    loss = tensorflow.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)
#consider adding more shrink layers, the final image is too big



training

In [ ]:
defect_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8
)

defect_model.save("drive/My Drive/Colab Notebooks/defect_mnist.h5")

Epoch 1/8
76/90 [========================>.....] - ETA: 2:26 - loss: 1.3899 - accuracy: 0.3956